In [1]:
import numpy as np
import time
m = 0.15

In [2]:
def create_COO_from_A(matrix):
    rows = []
    cols = []
    val = []
    n_page = matrix.shape[0]
    
    for i in range(0, n_page):
        for j in range(0, n_page):
            if matrix[i, j] != 0:
                val.append(matrix[i, j])
                rows.append(i)
                cols.append(j)
               
    val_array = np.array(val, dtype=np.float32) 
    rows_array = np.array(rows, dtype=np.int32)
    cols_array = np.array(cols, dtype=np.int32)
    return val_array, rows_array, cols_array, n_page

In [3]:
def power_method(row_index, column_index, out_degree, max_iter=100, tol=1.0e-8, m = 0.15):
    # Finds the dangling nodes
    dangling_mask = (out_degree == 0)
    
    # x guess vector initialization
    n_pages = out_degree.shape[0]
    x = np.ones(n_pages, dtype=np.float64) / n_pages
    
    # m*S
    randomness = m / n_pages
    e_current = 0
    
    print("Starting page rank:")
    start_time = time.time()
    for i in range(0, max_iter):
        x_prev = x.copy()
        
        # It's like doing A*x since that Aij = 1/out_degree(ij)
        first_comp = x[column_index] / out_degree[column_index]
        
        # Effective implementation of x = (1 - m)*Ax
        x = np.zeros(n_pages, dtype=np.float64)
        np.add.at(x, row_index, first_comp)
        x *= 1 - m
        
        # Final computation x = x + mS with attention to the dangling nodes
        dangling_sum = np.sum(x_prev[dangling_mask])
        rank_correction = randomness + ((1 - m) * dangling_sum / n_pages)
        x += rank_correction
        
        e_current = (x@x_prev) / (x_prev@x_prev)
        
        err = np.sum(np.abs(x - x_prev))
        if err < tol:
            print(f"Convergenza raggiunta all'iterazione {i}.")
            break
                
    print(f"Finished in {time.time() - start_time:.5f} seconds")
    return e_current, x

<h1>Exercise 1</h1>
<h4>
Suppose the people who own page 3 in the web of Figure 1 are infuriated by the fact that its importance score, computed using formula (2.1), is lower than the score of page 1. In an attempt to boost page 3’s score, they create a page 5 that links to page 3; page 3 also links to page 5. Does this boost page 3’s score above that of page 1?
</h4>
<p>Page 3 owners added page 5 which only points to page 3 and is pointed to only by page 3<br>
p1 -> p2, p3, p4<br>
p2 -> p3, p4<br>
p3 -> p1, p5<br>
p4 -> p1, p3<br>
p5 -> p3<br>
</p>
<p>
x1 = x3/2 + x4/2<br>
x2 = x1/3<br>
x3 = x1/3 + x2/2 + x4/2 + x5<br>
x4 = x1/3 + x2/2<br>
x5 = x3/2<br>
</p>
<p>
So we can compute the eigenvector using the power method:
</p>

In [4]:
A1 = np.array([[0, 0, 1/2, 1/2, 0],
              [1/3, 0, 0, 0, 0],
              [1/3, 1/2, 0, 1/2, 1],
              [1/3, 1/2, 0, 0, 0],
              [0, 0, 1/2, 0, 0]])

a1, v1 = np.linalg.eig(A1)
print(f"Eigenvalue calculated: {a1}")
print(f"Eigenvector calculated: {-v1[:, 0].real/np.sum(-v1[:, 0].real)}")

Eigenvalue calculated: [ 1.        +0.j          0.33633116+0.j         -0.30859587+0.27755748j
 -0.30859587-0.27755748j -0.71913942+0.j        ]
Eigenvector calculated: [0.24489796 0.08163265 0.36734694 0.12244898 0.18367347]


<p>As we can see, adding page 5 does increase page 3 score, making it the most important.

<h1>Exercise 3</h1>
<h4>Add a link from page 5 to page 1 in the web of Figure 2. The resulting web, considered as an undirected graph, is connected. What is the dimension of V1(A)?</h4>
<p>
p1 -> p2<br>
p2 -> p1<br>
p3 -> p4<br>
p4 -> p3<br>
p5 -> p1, p3, p4<br>
</p>
<p>
x1 = x2 + x5/3<br>
x2 = x1<br>
x3 = x4 + x5/3<br>
x4 = x3 + x5/3<br>
x5 = 0<br>
</p>

In [5]:
A3 = np.array([[0, 1, 0, 0, 1/3],
              [1, 0, 0, 0, 0],
              [0, 0, 0, 1, 1/3],
              [0, 0, 1, 0, 1/3],
              [0, 0, 0, 0, 0]])

x3, y3 = np.linalg.eig(A3)
print(x3)
print(f"Eigenvector associated to lamda1 = 1: {y3[:, 0].real/np.sum(y3[:, 0].real)}")
print(f"Eigenvector associated to lamda3 = 1: {y3[:, 2].real/np.sum(y3[:, 2].real)}")

[ 1. -1.  1. -1.  0.]
Eigenvector associated to lamda1 = 1: [0.5 0.5 0.  0.  0. ]
Eigenvector associated to lamda3 = 1: [0.  0.  0.5 0.5 0. ]


<p>
We can observe that dimension of V1(A) = 2, since there are 2 eigenvalues that equals 1 and therefore 2 possible different eigenvectors.
</p>

<h1>Exercise 5</h1>
<h4>Prove that in any web the importance score of a page with no backlinks is zero.</h4>
<p>
The importance score x_i of page i is given by the sum of the other pages' score, those who are pointing to page i, over the number of their outgoing links. If page i has no backlinks that means that no pages are pointing to page i, making the sum, and so its score, equal to 0.
</p>

<h1>Exercise 6</h1>
<h4>Implicit in our analysis up to this point is the assertion that the manner in which the pages of a web W are indexed has no effect on the importance score assigned to any given page, prove this.</h4>
<p>
Let W contains n pages, each page assigned an index 1 through n, and let A be the resulting link matrix. Suppose we then transpose the indices of pages i and j.<br>
We do this having A' = PAP where P is the elementary matrix obtained by transposing rows i and j of the n×n identity matrix.
</p>
<p>
Let's take for example A the matrix of the graph 2.1, one of his eigenvector was x = [12, 4, 9, 6].<br>
Now let's take P = [[1, 0, 0, 0], [0, 0, 1, 0], [0, 1, 0, 0], [0, 0, 0, 1]] so that it switches page 2 and 3.
let's compute A' = PAP
</p>

In [6]:
A6 = np.array([[0, 0, 1, 1/2],
              [1/3, 0, 0, 0],
              [1/3, 1/2, 0, 1/2],
              [1/3, 1/2, 0, 0]])

P = np.array([[1, 0, 0, 0],
              [0, 0, 1, 0],
              [0, 1, 0, 0],
              [0, 0, 0, 1]])

A6_tilde = P@A6@P
print(f"PAP:\n{A6_tilde}\n")

val6, row6, col6, n_pages6 = create_COO_from_A(A6)
val6_tilde, row6_tilde, col6_tilde, _ = create_COO_from_A(A6_tilde)

out6 = np.zeros(n_pages6, dtype=np.int32)
np.add.at(out6, col6, 1)
out6_tilde = np.zeros(n_pages6, dtype=np.int32)
np.add.at(out6_tilde, col6_tilde, 1)

e6, v6 = power_method(row6, col6, out6)
e6_tilde, v6_tilde = power_method(row6_tilde, col6_tilde, out6_tilde)

print(f"\nEigenvalue of A: {e6}")
print(f"Eigenvalue of PAP: {e6_tilde}")
print(f"\nEigenvector of A:\n{v6}")
print(f"Eigenvector of PAP:\n{v6_tilde}")

PAP:
[[0.         1.         0.         0.5       ]
 [0.33333333 0.         0.5        0.5       ]
 [0.33333333 0.         0.         0.        ]
 [0.33333333 0.         0.5        0.        ]]

Starting page rank:
Convergenza raggiunta all'iterazione 23.
Finished in 0.00110 seconds
Starting page rank:
Convergenza raggiunta all'iterazione 23.
Finished in 0.00100 seconds

Eigenvalue of A: 0.9999999997656641
Eigenvalue of PAP: 0.9999999997656642

Eigenvector of A:
[0.36815068 0.14180936 0.28796163 0.20207834]
Eigenvector of PAP:
[0.36815068 0.28796163 0.14180936 0.20207834]


The eigenvalues don't change and the eigenvectors are the same except for row 2 and 3 that are swapped.

<h1>Exercise 7</h1>
<h4>
Prove that if A is an n × n column-stochastic matrix and 0 ≤ m ≤ 1, then M=(1−m)A+mS is also a column-stochastic matrix.
</h4>
<p>
If we develop the sum for M each element will be in the form M[i, j] = (1 - m)*A[i, j] + m/n = A[i, j] - m*A[i, j] + m/n<br>
Now if we compute the sum on the column, so the sum from 1 to n, we obtain (we will denote 'sum' as the sum of the element in a column from index 0 to n):<br>
sum(A[i, j] - m*A[i, j] + m/n) = sum(A[i, j]) - sum(m*A[i, j]) + sum(m/n)<br>
but sum(A[i, j]) = 1 for A is column-stochastic, sum(m*A[i, j]) = m for the same reason and sum(m/n) = m so<br>
sum(A[i, j]) - sum(m*A[i, j]) + sum(m/n) = 1 - m + m = 1 and this proves that also M is column-stochastic.
</p>

<h1>Exercise 8</h1>
<h4>
Show that the product of two column-stochastic matrices is also column-stochastic.
</h4>
<p>
If we take two matrices A size (m x n) and B size (n x z) both column-stochastic and we compute the product we obtain:<br>
(we will refer as a00 the element of the matrix A in position (0, 0) and so for the other positions and for the other matrices, we will also use 'sum' for the sum along a certain column from indices 1 to m)<br>
S = A*B and if we analize the result element for element, for the first column, we get:<br>
s00 = a00*b00 + a01*b10 + ..... + a0n*bn0<br>
s10 = a10*b00 + a11*b10 + ..... + a1n*bn0<br>
.<br>
.<br>
.<br>
sm0 = an0*b00 + an1*b10 + ..... + ann*bn0<br><br>
It is easy to understand that the situation for the other columns is the same. Now let's do the sum of the first column, we can see that if we isolate every b element we get something like this:<br>
sum = b00(a00 + a10 + ... + an0) + b10(a01 + a11 + ... + an1) + ... + bm0(a0n + a1n + ... ann)<br><br>
But each sum of the a-elements is the sum along the column in the matrix A, therefore it equals to 1 per definition, leaving us with:<br>
sum = b00 + b10 + ... + bn0<br>
But this is also the sum of the b-elements along the columns of the matrix B, giving us for result 1 as well. This proves that the product of two column-stochastic matrices, even with different sizes, is also a column-stochastic matrix.
</p>

<h1>Exercise 9</h1>
<h4>
Show that a page with no backlinks is given importance score m/n by formula (3.2).
</h4>
<p>
As demonstrated in exercise 5 a page with no backlinks is given the importance score of 0 if we just consider A, but this remains the same if we do (1 - m)A since this does not change the 0-elements of the matrix.<br>
Now let's compute:<br>
x = (1 - m)Ax + mS = 0 + mS -> x = m/n
</p>

In [7]:
A9 = A3.copy()
S9 = np.full(A9.shape, 1/A9.shape[0])
M9 = (1-m)*A9 + m*S9
print(f"M:\n{M9}\n")

val9, row9, col9, n_pages9 = create_COO_from_A(A9)
out9 = np.zeros(n_pages9, dtype=np.int32)
np.add.at(out9, col9, 1)

e9, v9 = power_method(row9, col9, out9)
print(f"\nEigenvalues of M:{e9}")
print(f"Eigenvector of M:\n{v9}")

M:
[[0.03       0.88       0.03       0.03       0.31333333]
 [0.88       0.03       0.03       0.03       0.03      ]
 [0.03       0.03       0.03       0.88       0.31333333]
 [0.03       0.03       0.88       0.03       0.31333333]
 [0.03       0.03       0.03       0.03       0.03      ]]

Starting page rank:
Finished in 0.00536 seconds

Eigenvalues of M:0.9999999998869127
Eigenvector of M:
[0.23063063 0.22603604 0.25666667 0.25666667 0.03      ]


<h1>Exercise 10</h1>
<h4> Suppose that A is the link matrix for a strongly connected web of n pages (any page can be reached from any other page by following a finite number of links). Show that dim(V1(A)) = 1.</h4>
<p>
Let Aij,k be entry of row i and column j of the matrix A^k and when k is omitted it is considered to be 1. Then Aij > 0 if and only if page i can be reached from page j in exactly one step. Let's take matrix A from graph 2.1
</p>

In [8]:
A10 = np.array([[0, 0, 1, 1/2],
              [1/3, 0, 0, 0],
              [1/3, 1/2, 0, 1/2],
              [1/3, 1/2, 0, 0]])
print(f"Matrix A:\n{A10}\n")
print(f"Matrix A^2:\n{A10@A10}")

Matrix A:
[[0.         0.         1.         0.5       ]
 [0.33333333 0.         0.         0.        ]
 [0.33333333 0.5        0.         0.5       ]
 [0.33333333 0.5        0.         0.        ]]

Matrix A^2:
[[0.5        0.75       0.         0.5       ]
 [0.         0.         0.33333333 0.16666667]
 [0.33333333 0.25       0.33333333 0.16666667]
 [0.16666667 0.         0.33333333 0.16666667]]


In [9]:
print(A10.shape[0])
I = np.eye(4)
print(I)

4
[[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]]


In [10]:
print(np.sum((I + A10), axis=0))
print(np.sum((I + A10 + A10@A10), axis=0))
print(np.sum((I + A10 + A10@A10 + A10@A10@A10), axis=0))

[2. 2. 2. 2.]
[3. 3. 3. 3.]
[4. 4. 4. 4.]


<p>
This shows how each page is reachable from another one in exactly two steps, for example page 1 can reach page 1 going to page 3 and back, meanwhile page 3 can not reach page 1 in two steps so its entry A[0, 2] = 0.
</p>
<p>
(We'll define sum() as the sum from 0 to k)<br>
Since we have that Aij,2 = sum(Aik*Akj) so if Aij,2 > 0 implies that for some k both Aik and Akj are positive, so a path in two steps exists. The same logic can be applied to A^3 to find which page can reach/be reached another one in exactly 3 steps, and more general if Aij,p > 0 that means that page i can be reached from page j in exactly p steps.
</p>
<p>
With this consideration follows that (I + A + A^2 + ... + A^p)ij > 0 if page i can be reached from page j in p or fewer steps (we include I as A^0, to show that each page can reach itself in 0 steps). Moreover we can see that A,pos = (I + A + A^2 + ... + A^n-1) is a positive matrix if the web is strongly connected, because if every page can reach every other page it is assured that at least one ij-entry in the sum of the matrices is != 0, the worst case scenario is that page i can reach exactly page j in n-1 steps.
</p>
<p>
If we then take B = A,pos / n we obtain a positive matrix who is also column-stochastic.
</p>

In [11]:
B10 = (I + A10 + A10@A10 + A10@A10@A10)/A10.shape[0]
print(f"Matrix B:\n{B10}")
print(np.sum(B10, axis=0))

Matrix B:
[[0.47916667 0.25       0.375      0.3125    ]
 [0.125      0.3125     0.08333333 0.08333333]
 [0.22916667 0.25       0.41666667 0.25      ]
 [0.16666667 0.1875     0.125      0.35416667]]
[1. 1. 1. 1.]


In [12]:
e1, x1 = np.linalg.eig(A10)
e2, x2 = np.linalg.eig(B10)

ex1 = x1[:, 0].real/np.sum(x1[:, 0].real)
ex2 = -x2[:, 0].real/np.sum(-x2[:, 0].real)

print(f"Eigenvector of B:\n{ex1}")
print(f"Eigenvector of A:\n{ex2}")

Eigenvector of B:
[0.38709677 0.12903226 0.29032258 0.19354839]
Eigenvector of A:
[0.38709677 0.12903226 0.29032258 0.19354839]


<p>
This works beacuse B is a polinomial combination of A, implying that if v is the eigenvector of A with respects to the eigenvalue e then v is also the eigenvector of B with respect to the polinomial combination of e.
</p>
<p>
We can conclude that if x is present in both A and B, given that A and B are positive and column-stochastic, then dim(V1(B)) = dim(V1(A)) = 1.
</p>

<h1>Exercise 11</h1>
<h4>Consider again the web in Figure 2.1, with the addition of a page 5 that links to page 3, where page 3 also links to page 5. Calculate the new ranking by finding the eigenvector of M(corresponding to λ = 1) that has positive components summing to one. Use m = 0.15.</h4>

In [13]:
A11 = np.array([[0, 0, 1/2, 1/2, 0],
              [1/3, 0, 0, 0, 0],
              [1/3, 1/2, 0, 1/2, 1],
              [1/3, 1/2, 0, 0, 0],
              [0, 0, 1/2, 0, 0]])
print(f"Matrix A:\n{A11}")

Matrix A:
[[0.         0.         0.5        0.5        0.        ]
 [0.33333333 0.         0.         0.         0.        ]
 [0.33333333 0.5        0.         0.5        1.        ]
 [0.33333333 0.5        0.         0.         0.        ]
 [0.         0.         0.5        0.         0.        ]]


In [14]:
m11 = 0.15
S11 = np.full((A11.shape[0], A11.shape[0]), 1/A11.shape[0])
M11 = ((1 - m11)*A11) + (m11*S11)
print(f"M:\n{M11}\n")

val11, row11, col11, n_pages11 = create_COO_from_A(A11)

out11 = np.zeros(n_pages11, dtype=np.int32)
np.add.at(out11, col11, 1)

e11, v11 = power_method(row11, col11, out11)
print(f"\nEigenvalues of M:{e11}")
print(f"Eigenvector of M:\n{v11}")

M:
[[0.03       0.03       0.455      0.455      0.03      ]
 [0.31333333 0.03       0.03       0.03       0.03      ]
 [0.31333333 0.455      0.03       0.455      0.88      ]
 [0.31333333 0.455      0.03       0.03       0.03      ]
 [0.03       0.03       0.455      0.03       0.03      ]]

Starting page rank:
Convergenza raggiunta all'iterazione 37.
Finished in 0.00229 seconds

Eigenvalues of M:0.9999999989965619
Eigenvector of M:
[0.23714058 0.09718983 0.34889409 0.13849551 0.17827999]


<p>
We can see that with the addiction of page 5 page 3 becomes the most important.
</p>

<h1>Exercise 12</h1>
<h4>Add a sixth page that links to every page of the web in the previous exercise, but to which no other page links. Rank the pages using A, then using M with m = 0.15, and compare the results.</h4>

In [15]:
A12 = np.array([[0, 0, 1/2, 1/2, 0, 1/5],
              [1/3, 0, 0, 0, 0, 1/5],
              [1/3, 1/2, 0, 1/2, 1, 1/5],
              [1/3, 1/2, 0, 0, 0, 1/5],
              [0, 0, 1/2, 0, 0, 1/5],
              [0, 0, 0, 0, 0, 0]])
m12 = 0.15
S12 = np.full((A12.shape[0], A12.shape[0]), 1/A12.shape[0])
M12 = (1-m12)*A12 + (m12*S12)
print(f"M:\n{M12}\n")

val12, row12, col12, n_pages12 = create_COO_from_A(A12)

out12 = np.zeros(n_pages12, dtype=np.int32)
np.add.at(out12, col12, 1)

e12_1, x12_1 = np.linalg.eig(A12)
e12_2, v12_2 = power_method(row12, col12, out12)

print(f"Eigenvalues of A:\n{e12_1}\n")
print(f"Eigenvalue of M: {e12_2}\n")

print(f"Eigenvector of A:\n{x12_1[:, 0].real/np.sum(x12_1[:, 0].real)}")
print(f"Eigenvector of M:\n{v12_2}\n")

M:
[[0.025      0.025      0.45       0.45       0.025      0.195     ]
 [0.30833333 0.025      0.025      0.025      0.025      0.195     ]
 [0.30833333 0.45       0.025      0.45       0.875      0.195     ]
 [0.30833333 0.45       0.025      0.025      0.025      0.195     ]
 [0.025      0.025      0.45       0.025      0.025      0.195     ]
 [0.025      0.025      0.025      0.025      0.025      0.025     ]]

Starting page rank:
Convergenza raggiunta all'iterazione 36.
Finished in 0.00295 seconds
Eigenvalues of A:
[ 1.        +0.j          0.33633116+0.j         -0.30859587+0.27755748j
 -0.30859587-0.27755748j -0.71913942+0.j          0.        +0.j        ]

Eigenvalue of M: 1.0000000010100603

Eigenvector of A:
[ 0.24489796  0.08163265  0.36734694  0.12244898  0.18367347 -0.        ]
Eigenvector of M:
[0.23121206 0.09476009 0.34017174 0.13503312 0.17382299 0.025     ]



<p>
As we can see the addiction of page 6 doesn't really change the eigenvector that much, the pages' score barely changes and the most important page is still page 3.
</p>

<h1>Exercise 13</h1>
<h4>Construct a web consisting of two or more subwebs and determine the ranking given by formula M = (1-m)A + mS.</h4>
<p>
We will use m = 0.15, our web will consist of 3 subwebs:<br><br>
Subweb 1<br>
page 1 -> page 2<br>
page 2 -> page 3<br>
page 3 -> page 1<br><br>
Subweb 2<br>
page 4 -> page 5, page 6<br>
page 5 -> page 7<br>
page 6 -> page 5, page 7<br>
page 7 -> page 4<br><br>
Subweb 3<br>
page 8 -> page 9, page 10<br>
page 9 -> page 8, page 11<br>
page 10 -> page 8<br>
page 11 -> page 8, page 9, page 10<br>
</p>

<img src="graph_full.png">

In [16]:
A13 = np.array([[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
                [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
                [0, 0, 0, 1/2, 0, 1/2, 0, 0, 0, 0, 0],
                [0, 0, 0, 1/2, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 1, 1/2, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 1/2, 1, 1/3],
                [0, 0, 0, 0, 0, 0, 0, 1/2, 0, 0, 1/3],
                [0, 0, 0, 0, 0, 0, 0, 1/2, 0, 0, 1/3],
                [0, 0, 0, 0, 0, 0, 0, 0, 1/2, 0, 0]])
print(f"Matrix A:\n{A13}")
print(np.sum(A13, axis=0))

Matrix A:
[[0.         0.         1.         0.         0.         0.
  0.         0.         0.         0.         0.        ]
 [1.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.        ]
 [0.         1.         0.         0.         0.         0.
  0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.         0.
  1.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.5        0.         0.5
  0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.5        0.         0.
  0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         1.         0.5
  0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.         0.
  0.         0.         0.5        1.         0.33333333]
 [0.         0.         0.         0.       

In [17]:
M13 = (1-m)*A13 + m/A13.shape[0]

val13, row13, col13, n_pages13 = create_COO_from_A(A13)

out13 = np.zeros(n_pages13, dtype=np.int32)
np.add.at(out13, col13, 1)

e13, x13 = power_method(row13, col13, out13)

print(f"\nEigenvalue of M:{e13}")
print(f"Eigenvector of M:\n{x13}")

Starting page rank:
Convergenza raggiunta all'iterazione 63.
Finished in 0.00429 seconds

Eigenvalue of M:0.9999999986787343
Eigenvector of M:
[0.09090909 0.09090909 0.09090909 0.10807628 0.08488552 0.05956878
 0.11110578 0.13894645 0.08703239 0.08703239 0.05062513]


<p>
As we can see for subweb 1 (eigenvector's elements 1 to 3) there's not a most important page, as every page points to another page in a circle, for subweb 2 (eigenvector's elements 4 to 7) the most important is page 7 and for subweb 3 (eigenvector's elements 8 to 11) it's page 8.<br>
Overall the most important page in this web is page 8.
</p>

<h1>Exercise 14</h1>
<h4>For the web in Exercise 11, compute the values of ||M^k * x0 − q||1 and (||M^k * x0 − q||1)/(||M^(k−1) * x0 − q||1) for k = 1, 5, 10, 50, using an initial guess x0 not too close to the actual eigenvector q. Determine c = max, 1<=j<=n, |1 − 2 * min, 1<=i<=n, Mij| and the absolute value of the second largest eigenvalue of M.</h4>
<p>
In this exercise, since our method converges really fast, we increase the tolerance so that our results will be as precise as possible, since the more precise the eigenvector is the less our cycle is likely to diverge from the result, this due to approximation errors.
</p>

In [18]:
A14 = A11.copy()
A14 = A14.astype(np.float64)
M14 = (1 - m)*A14 + m/A14.shape[0]
print(f"M:\n{M14}\n")

val14, row14, col14, n_pages14 = create_COO_from_A(A14)

out14 = np.zeros(n_pages14, dtype=np.int32)
np.add.at(out14, col14, 1)

e14, v14 = power_method(row14, col14, out14, tol=1.0e-15)

e14_calc, v14_calc = np.linalg.eig(M14)
v14 = v14/np.sum(v14)
v14_calc = (v14_calc[:, 0]/np.sum(v14_calc[:, 0])).real

print(f"\nEigenvalues of M linalg:\n{e14_calc}")
print(f"Eigenvalue M power method: {e14}\n")

print(f"Eigenvector linalg.eig:\n{v14_calc}")
print(f"Eigenvector power method:\n{v14}")
print(f"Error between Eigenvectors: {np.linalg.norm(v14 - v14_calc)}")

M:
[[0.03       0.03       0.455      0.455      0.03      ]
 [0.31333333 0.03       0.03       0.03       0.03      ]
 [0.31333333 0.455      0.03       0.455      0.88      ]
 [0.31333333 0.455      0.03       0.03       0.03      ]
 [0.03       0.03       0.455      0.03       0.03      ]]

Starting page rank:
Convergenza raggiunta all'iterazione 70.
Finished in 0.00300 seconds

Eigenvalues of M linalg:
[ 1.        +0.j          0.28588149+0.j         -0.26230649+0.23592386j
 -0.26230649-0.23592386j -0.61126851+0.j        ]
Eigenvalue M power method: 1.0000000000000002

Eigenvector linalg.eig:
[0.23714058 0.09718983 0.34889409 0.13849551 0.17827999]
Eigenvector power method:
[0.23714058 0.09718983 0.34889409 0.13849551 0.17827999]
Error between Eigenvectors: 3.3074585536510505e-16


In [19]:
x0 = np.zeros(A14.shape[0], dtype=np.float64)
x0[0] = 1.0

k_values = [1, 5, 10, 50]

current_x = x0.copy()
error_prev = np.linalg.norm(x0 - v14, ord=1)
anda2 = 0

print(f"{'k':<5} | {'Error ||Mk*x0 - q||1':<25} | {'Ratio (Err_k / Err_k-1)':<25}")
print("-" * 60)

for k in range(min(k_values), max(k_values)+1):
    current_x = M14 @ current_x
    
    error_curr = np.linalg.norm(current_x - v14, ord=1)
    ratio = error_curr / error_prev if error_prev > 1e-16 else 0.0
    
    print(f"{k:<5} | {error_curr:<25.10f} | {ratio:<25.5f}")
    
    error_prev = error_curr
    anda2 = ratio

min_vals_col = np.min(M14, axis=0)
c_vals = np.abs(1 - 2 * min_vals_col)
c = np.max(c_vals)

print("\n" + "="*30)
print(f"Valore di c: {c:.4f}")
print(f"Modulo secondo autovalore (|lambda_2|): {anda2:.5f}")
print(f"Eigenvalues:\n{e14_calc}")

k     | Error ||Mk*x0 - q||1      | Ratio (Err_k / Err_k-1)  
------------------------------------------------------------
1     | 0.7819626529              | 0.51252                  
2     | 0.1847278214              | 0.23624                  
3     | 0.0459614000              | 0.24881                  
4     | 0.0212929306              | 0.46328                  
5     | 0.0068714107              | 0.32271                  
6     | 0.0022902823              | 0.33331                  
7     | 0.0014104269              | 0.61583                  
8     | 0.0009566879              | 0.67830                  
9     | 0.0005802764              | 0.60655                  
10    | 0.0003043717              | 0.52453                  
11    | 0.0001660407              | 0.54552                  
12    | 0.0000955945              | 0.57573                  
13    | 0.0000583056              | 0.60993                  
14    | 0.0000361713              | 0.62037                  
15    | 0

<p>
As we can see the method converges pretty fast, already at iteration 15 we have a good representation of |λ2|, in the end we obtain 0.61213, which is really close to the one obtained with np.linalg.eig which is 0.61126.<br>
Let's compute then c = 1 - 2*m/n = 1 - 2*0.15/5 = 0.94 the upper bound of c, we can see that with our formula we obtained the highest possible value.
</p>

<h1>Exercise 16</h1>
<h4>Given matrix A show that M = (1 − m)A + mS (all Sij = 1/3) is not diagonalizable for 0 ≤ m < 1.</h4>
To show that M is diagonalizable we must prove that for every eigenvalue the algebraic multiplicity equals the geometric multiplicity.

In [20]:
A16 = np.array([[0, 1/2, 1/2], [0, 0, 1/2], [1, 1/2, 0]])
S16 = np.full((A16.shape[0], A16.shape[0]), 1/A16.shape[0])
print(f"A:\n{A16}\n")
print(f"S:\n{S16}")

A:
[[0.  0.5 0.5]
 [0.  0.  0.5]
 [1.  0.5 0. ]]

S:
[[0.33333333 0.33333333 0.33333333]
 [0.33333333 0.33333333 0.33333333]
 [0.33333333 0.33333333 0.33333333]]


Now let's build M for a generic m M = (1 - m)A + mS so we will have:

In [21]:
M16 = [[    m/3,      (1-m)/2 + m/3,  (1-m)/2 + m/3],
       [    m/3,            m/3,      (1-m)/2 + m/3],
       [(1-m) + m/3,  (1-m)/2 + m/3,        m/3]]

M16 = [[   m/3,     (3-m)/6,  (3-m)/6],
       [   m/3,       m/3,    (3-m)/6],
       [(3-2*m)/6,  (3-m)/6,     m/3]]

<p>
Let's calculate the characteristic polynomial as p(λ) = det(M - λ I) = (λ - 1) * (λ - (m-1)/2)^2<br>
As we can see the eigenvalues (1, (m-1)/2) have algebraic multiplicity of (1, 2), let's see if the geometric multiplicity is also (1, 2), otherwise M is not diagonalizable.<br>
- λ = 1<br>
Since 1 <= mg <= ma and ma = 1 then mg also = 1.<br>
- λ = (1 - m)/2<br>
We can compute the geometric multiplicity as:<br>
mg = dim - rank(D) = 3 - rank(M - (m - 1)/2 * I)
</p>

In [22]:
D16 = [[ (3-m)/6,   (3-m)/6, (3-m)/6],
       [   m/3,     (3-m)/6, (3-m)/6],
       [(3-2*m)/3,  (3-m)/6, (3-m)/6]]

<p>
If we look at the columns of D we can see that column 2 and 3 are identical and therefore L.D., if we compare instead column 1 and 2 we can see that they are L.D if and only if m = 1.<br><br>
But in our premise we say that it must be m<1, therefore column 1 and column 2 must be L.I. giving us rank(D) = 2 and mg = 3 -rank(D) = 1 which is different from ma = 2.<br><br>
We can conclude that, for the link matrix A and 0 <= m < 1, M is not diagonalizable.
</p>

<h1>Exercise 17</h1>
<h4>How should the value of m be chosen? How does this choice affect the rankings and the computation time?</h4>
<p>
There's not exactly a right choice for m, its value can be decided by different approaches:<br>
<ul>
    <li>Matrixm m and rankings
    <li>Computation time 
</ul>
</p>
<b><u>Matrix m and rankings</u></b><br>
<p>
One way it's to analyse M and how it's built, M = (1 - m)A + mS if we choose a low value of m this means that we are giving more importance to the link matrix A and the rankings would depend heavily on that, instead if we choose a higher value it means that the matrix A is not that important giving us a ranking more uniform.<br>
Let's take the example of the web-surfer, the surfer starts at a random page and randomly moves to another one according to the following procedure: if the surfer is currently at a page with r outgoing links, he either randomly chooses anyone of these links with probability (1 − m)/r or he jumps to any randomly selected page on the web, each with probability m/n. Let's analyze different scenarios:<br>
</p>
<b>- Lower m</b><br>
<p>
If we choose a low value for m, it gives us a higher probability of the surfer to follow the links through the pages, this is because m boosts the importance of A in the formula of M and lowers the importance of S.<br>
Be aware that a really low m might create some trouble in the randomness of a web search, if m is 0 we end up in the case where the surfer only follows the pages' links, but it might get trapped in a circle following the same links, leaving some pages unvisited. It practically kills the randomness of the process.<br>
</p>
<b>- Higher m</b><br>
<p>
On the other hand if we assign m a high value it significantly lowers the importance of the link matrix A, if for example m = 1 then M = S completely disregarding the matrix A. In the example of the web surfer this means that the probability of it jumping to a random web page, rather than following links between the pages, is very high.<br>
This might cause problems in the opposite way as before, a high m might completely ignore the links between pages making the overall structure useless, since it moves randomly.
</p>
<br><b><u>Computation time</u></b><br>
<p>
Another way is to consider the computation time, since it is demonstrated that the power method converges asymptotically according |M*x_k - q|1 == |λ2|*|x - q|1, this means that the lower λ2 is, the faster the method converges. But it's also known that |λ2| <= 1 - m and this gives us two scenarios over the value of m: high m -> the method will converge really fast, low m -> method will converge really slow.
</p>
<p>
<br><b>So, how should we choose m?</b><br>
If we go for a high value we get a faster computation time but also a higher randomness of the web, lowering the importance of the pages' links.<br>
If we go for a lower value we get a slower computation time but we give more importance to A, therefore obtaining the behavior of following the links that were given.<br>
The value of m originally used by Google is reportedly 0.15 and this is is a good compromise between speed and importance of A.
</p>